<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Coarse_Grained_Simulations_with_Martini_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Standard MD simulations, while incredibly valuable, can be computationally intensive and time-consuming, often limiting our ability to study long-timescale processes in biology.

To overcome these limitations, coarse-grained (CG) simulations have emerged as a powerful solution.

The basic principle behind CG simulations is the mapping of multiple atoms to a single representative entity, often referred to as a bead. By doing so, we have different advantages over atomistic simulations. Here I will only mention the two that are more intuitive:

**Reduced Atom Count**: In CG simulations, multiple atoms are mapped to a single representative entity, resulting in a significantly reduced number of atoms in the system. This reduction in atom count allows for more efficient computations and faster simulations.

**Larger Timesteps**: The simplification of the system in CG simulations also enables the use of larger timesteps compared to atomistic simulations. With fewer degrees of freedom to consider, CG simulations can effectively explore longer timescales, potentially up to 20 fs or even more. This larger timestep further contributes to the computational speedup of CG simulations.


The general workflow is not particularly different from a standard MD simulation. The only thing we need to change is the initial step since the protein will need to be converted from an atomistic structure to a coarse-grained one but the rest will be pretty much similar to a standard MD simulation

In [ ]:
# https://www.compchems.com/gromacs-tutorial-coarse-grained-simulations-with-martini-3-force-field/#cg-simulations-workflow
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!conda --version

In [ ]:
import condacolab
condacolab.check()

In [ ]:
!pip install -q numpy py3Dmol


Define the Martini Beads and Mapping:

The Martini model simplifies groups of atoms into single beads.

Here's an example of how to define and map atoms to Martini beads.

In [ ]:
import numpy as np

# Define atoms of a molecule
atoms = {
    'atom1': [1.0, 2.0, 3.0],
    'atom2': [2.0, 3.0, 4.0],
    'atom3': [3.0, 4.0, 5.0],
    'atom4': [4.0, 5.0, 6.0]
}

# Define mapping of atoms to Martini beads
mapping = {
    'bead1': ['atom1', 'atom2'],
    'bead2': ['atom3', 'atom4']
}

# Calculate the position of each Martini bead as the center of mass of the mapped atoms
beads = {}
for bead, atoms_list in mapping.items():
    positions = np.array([atoms[atom] for atom in atoms_list])
    beads[bead] = np.mean(positions, axis=0)

print(beads)


Visualize the Coarse-Grained Structure with Py3Dmol

In [ ]:
import py3Dmol

# Create Py3Dmol viewer
view = py3Dmol.view(width=800, height=600)

# Add atoms as spheres
for atom, pos in atoms.items():
    view.addSphere({'center': {'x': pos[0], 'y': pos[1], 'z': pos[2]}, 'radius': 0.3, 'color': 'red'})

# Add beads as larger spheres
for bead, pos in beads.items():
    view.addSphere({'center': {'x': pos[0], 'y': pos[1], 'z': pos[2]}, 'radius': 0.5, 'color': 'blue'})

# Show the visualization
view.show()


Coarse-Graining with FENE Interactions

FENE (Finite Extensible Nonlinear Elastic) interactions are commonly used in coarse-grained models to represent bonded interactions with nonlinear elasticity.

In [ ]:
import matplotlib.pyplot as plt

def fene_potential(r, k=30.0, R0=1.5):
    """Calculate the FENE potential."""
    return -0.5 * k * R0**2 * np.log(1 - (r / R0)**2)

# Plot the FENE potential
r = np.linspace(0, 1.5, 100)
potential = fene_potential(r)

plt.plot(r, potential)
plt.xlabel('Distance (r)')
plt.ylabel('FENE Potential')
plt.title('FENE Potential Curve')
plt.show()


Apply FENE Interactions in a Coarse-Grained Simulation

In [ ]:
# Define initial positions of coarse-grained beads
beads_positions = np.array([
    [1.5, 2.5, 3.5],
    [3.5, 4.5, 5.5]
])

# Define FENE parameters
k = 30.0
R0 = 1.5

# Function to calculate FENE force
def fene_force(r, k=30.0, R0=1.5):
    """Calculate the FENE force."""
    return -k * r / (1 - (r / R0)**2)

# Simple simulation loop
timesteps = 100
positions = [beads_positions.copy()]
for _ in range(timesteps):
    r = np.linalg.norm(beads_positions[1] - beads_positions[0])
    force = fene_force(r, k, R0)
    direction = (beads_positions[1] - beads_positions[0]) / r
    beads_positions[0] += force * direction * 0.01
    beads_positions[1] -= force * direction * 0.01
    positions.append(beads_positions.copy())

# Plot the positions of the beads over time
positions = np.array(positions)
plt.plot(positions[:, 0, 0], positions[:, 0, 1], '-o',label='Bead 1')
plt.plot(positions[:, 1, 0], positions[:, 1, 1], '-s',label='Bead 2')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Bead Positions Over Time')
plt.legend()
plt.show()


More involved example

In [ ]:
!conda install -c salilab dssp -y --quiet

In [ ]:
!conda install gromacs -y --quiet

In [ ]:
pip install -q vermouth

In [ ]:
pip install -q git+https://github.com/marrink-lab/vermouth-martinize.git#vermouth


In [ ]:
pip install -q insane

First of all, we can download an atomistic structure from the Protein Data Bank. As an example, we are going to use the famous lysozyme (PDB ID: 1AKI)

In [ ]:
!wget http://www.pdb.org/pdb/files/1AKI.pdb


The first thing to do as always is to make sure to clean the protein from other molecules. You can use the grep command

In [ ]:
!grep -v HETATM 1AKI.pdb > 1aki_clean.pdb


In [ ]:
!ls

In [ ]:
!which mkdssp

Check martinize2 is installed

In [ ]:
!martinize2 -h


In [ ]:
import py3Dmol
with open("1AKI.pdb") as ifile:
    system = "".join([x for x in ifile])
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()

In [ ]:
import py3Dmol
with open("1aki_clean.pdb") as ifile:
    system = "".join([x for x in ifile])
view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(system)
view.setStyle({'model': -1}, {"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.show()

To convert the clean atomistic structure of the protein to CG and generate the corresponding topology file we can use the Martinize

In [ ]:
!martinize2 -f 1aki_clean.pdb -x 1aki_cg.pdb -o topol.top -ff martini3001 -scfix -cys auto -p backbone -elastic -ef 700.0 -el 0.5 -eu 0.9


In [ ]:
# # !martinize2 -f 1aki_clean.pdb -dssp /usr/local/bin/mkdssp -x 1aki_cg.pdb -o topol.top -ff martini3001 -scfix -cys auto -p backbone -elastic -ef 700.0 -el 0.5 -eu 0.9
# !martinize2 -f 1aki_clean.pdb  -x 1aki_cg.pdb -o topol.top -ff martini3001 -scfix -cys auto -p backbone -elastic -ef 700.0 -el 0.5 -eu 0.9


After obtaining our coarse-grained (CG) structure, we need to prepare the system for the simulation. This process typically involves three essential steps: solvation, ion addition for system neutrality, and optionally embedding the protein in a membrane environment.

In [ ]:
!insane -f 1aki_cg.pdb -o system.gro -p topol.top -d 7 -sol W:100 -salt 0.15


Inside here you will find the general parameters of the force field, as well as the ones for the water and the ions.

In [ ]:
!wget https://www.compchems.com/martini_itp.tar.gz -O martini_itp.tar.gz

Untar file

In [ ]:
!tar -xvf martini_itp.tar.gz


In [ ]:
"""
vi martini_itp/martini_v3.0.0_solvents_v1.itp

;;;;; Martini 3 - Database of solvents
;;;;;
;;;;; File updated on 2021-03-29
;;;;;
;;;;; Version: 3.0.0_v1
;;;;;
;;;;; This collection contains the models included as part of the Martini 3 publication:
;;;;;   PCT Souza, et al., Nat. Methods, 2021. DOI: 10.1038/s41592-021-01098-3

;;;;;; WATER (representing 4 molecules)

[ moleculetype ]
; molname       nrexcl
  W             1

[ atoms ]
;id     type    resnr   residu  atom    cgnr    charge
 1      W      1        W       W      1       0

;;;;;;; DIMETHYLSULFOXIDE

[ moleculetype ]
; molname       nrexcl
  DMSO          1

[atoms]
; id    type    resnr   residu  atom    cgnr    charge
  1     SC6     1       DMS     S1      1       0
  2     TP6     1       DMS     O2      2       0

"""

In [ ]:
!ls -altr

In [ ]:
!mv molecule_0.itp martini_itp


In [ ]:
!ls -altr

We need to modify the topology file (topol.top) to include all the parameters we need for our simulation. This is what the file should be like:

In [ ]:
lines="""#include "martini_itp/martini_v3.0.0.itp"
#include "martini_itp/martini_v3.0.0_solvents_v1.itp"
#include "martini_itp/martini_v3.0.0_ions_v1.itp"
#include "martini_itp/molecule_0.itp"

[ system ]
; name
Insanely solvated protein.

[ molecules ]
; name  number
molecule_0       1
W            10672
NA+            113
CL-            121
"""
with open('topol.top',"w") as ifile:
    ifile.write(lines)

In [ ]:
!ls -altr

In [ ]:
import os
os.makedirs("minim")
os.chdir("minim")


In [ ]:
!pwd

In [ ]:
!wget https://www.compchems.com/gromacs_cg/minim.mdp -O minim.mdp

In [ ]:
!gmx grompp -f minim.mdp -c ../system.gro -r ../system.gro -p ../topol.top -o em.tpr


In [ ]:
!gmx mdrun -v -deffnm em


.gro coordinates are in nm, whereas *.pdb coordinates in Å
https://colab.research.google.com/github/lorenzopallante/BiomeccanicaMultiscala/blob/main/LAB/06-Gromacs/06-Gromacs.ipynb

In [ ]:
!ls -altr

In [ ]:
!echo "9 0" | gmx energy -f em.edr -o example_property.xvg
import matplotlib.pyplot as plt
import numpy as np

x,y = np.loadtxt("example_property.xvg",comments=["@", "#"],unpack=True)
plt.plot(x,y)
plt.xlabel("time (ps)")
plt.ylabel("Potential")
plt.show()

In [ ]:
!echo "6 0" | gmx energy -f em.edr -o example_property.xvg
import matplotlib.pyplot as plt
import numpy as np

x,y = np.loadtxt("example_property.xvg",comments=["@", "#"],unpack=True)
plt.plot(x,y)
plt.xlabel("time (ps)")
plt.ylabel("LJ-(SR)")

plt.show()

In [ ]:
!echo "7 0" | gmx energy -f em.edr -o example_property.xvg
import matplotlib.pyplot as plt
import numpy as np

x,y = np.loadtxt("example_property.xvg",comments=["@", "#"],unpack=True)
plt.plot(x,y)
plt.xlabel("time (ps)")
plt.ylabel("Coulomb (SR)")
plt.savefig("volume.png", format="png", dpi=300)
plt.show()

In [ ]:
pip install numpy py3Dmol


In [ ]:
import numpy as np

# Define atoms of a molecule
atoms = {
    'atom1': [1.0, 2.0, 3.0],
    'atom2': [2.0, 3.0, 4.0],
    'atom3': [3.0, 4.0, 5.0],
    'atom4': [4.0, 5.0, 6.0]
}

# Define mapping of atoms to Martini beads
mapping = {
    'bead1': ['atom1', 'atom2'],
    'bead2': ['atom3', 'atom4']
}

# Calculate the position of each Martini bead as the center of mass of the mapped atoms
beads = {}
for bead, atoms_list in mapping.items():
    positions = np.array([atoms[atom] for atom in atoms_list])
    beads[bead] = np.mean(positions, axis=0)

print(beads)


In [ ]:
import py3Dmol

# Create Py3Dmol viewer
view = py3Dmol.view(width=800, height=600)

# Add atoms as spheres
for atom, pos in atoms.items():
    view.addSphere({'center': {'x': pos[0], 'y': pos[1], 'z': pos[2]}, 'radius': 0.3, 'color': 'red'})

# Add beads as larger spheres
for bead, pos in beads.items():
    view.addSphere({'center': {'x': pos[0], 'y': pos[1], 'z': pos[2]}, 'radius': 0.5, 'color': 'blue'})

# Show the visualization
view.show()


In [ ]:
import matplotlib.pyplot as plt

def fene_potential(r, k=30.0, R0=1.5):
    """Calculate the FENE potential."""
    return -0.5 * k * R0**2 * np.log(1 - (r / R0)**2)

# Plot the FENE potential
r = np.linspace(0, 1.5, 100)
potential = fene_potential(r)

plt.plot(r, potential)
plt.xlabel('Distance (r)')
plt.ylabel('FENE Potential')
plt.title('FENE Potential Curve')
plt.show()


In [ ]:
# Define initial positions of coarse-grained beads
beads_positions = np.array([
    [0.0, 0.0, 0.0],
    [1.0, 1.0, 1.0]
])

# Define FENE parameters
k = 30.0
R0 = 1.5

# Function to calculate FENE force
def fene_force(r, k=30.0, R0=1.5):
    """Calculate the FENE force."""
    return -k * r / (1 - (r / R0)**2)

# Simple simulation loop
timesteps = 100
positions = [beads_positions.copy()]
for _ in range(timesteps):
    r = np.linalg.norm(beads_positions[1] - beads_positions[0])
    force = fene_force(r, k, R0)
    direction = (beads_positions[1] - beads_positions[0]) / r
    beads_positions[0] += force * direction * 0.01
    beads_positions[1] -= force * direction * 0.01
    positions.append(beads_positions.copy())

# Plot the positions of the beads over time
positions = np.array(positions)
plt.plot(positions[:, 0, 0], positions[:, 0, 1], label='Bead 1')
#plt.plot(positions[:, 1, 0], positions[:, 1, 1], label='Bead 2')
plt.xlabel('X Position')
plt.ylabel('Y Position')
plt.title('Bead Positions Over Time')
plt.legend()
plt.show()
